# Trajectory adapter on LTX-Video 2B: one notebook, run top to bottom
Open in VS Code, **Select Kernel -> Colab -> New Colab Server**, pick an **A100**, run the cells in order.
Needs `ie643_colab.zip` (code + 714 clips with masks and captions) in the root of **My Drive**.
Everything is written to `My Drive/ie643_ckpt/<RUN>`, so a disconnect loses nothing (re-run the cells; training resumes).

| stage | what it shows | rough time on an A100 (estimates, not measured) |
|---|---|---|
| 1 | plain 2B with its per-clip prompt, and the **training-free** drag-video start (no training needed) | 10 min |
| 2 | train the adapter (12 epochs) | 50-60 min |
| 3 | quick visual grid with the trained adapter | 10 min |
| 4 | benchmark: 3 variants, 2 seeds, metrics incl. background motion | 60-90 min (FAST: 15 min) |
| 5 | optional: clean out-of-domain images | 15 min |
| 6 | optional: focus-loss ablation (the instructor's suggested experiment) | 1.5 h |

**Use a NEW `RUN` name whenever the code or config changes**, otherwise a run resumes the old weights.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "pick a GPU runtime"
assert torch.cuda.get_device_properties(0).total_memory > 30e9, "use an A100 (the 2B model at 480x864 needs more than a T4/L4)"

In [ ]:
import os, glob
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):   # not mounted yet
    # 'Mountpoint must not already contain files' = something wrote to /content/drive before the mount (or an old half-mount):
    # Runtime -> Disconnect and delete runtime, then run THIS cell first. Never rm -rf anything under /content/drive.
    drive.mount('/content/drive')
if not os.path.isdir('/content/ie643'):
    zips = glob.glob('/content/drive/MyDrive/ie643_colab*.zip') + glob.glob('/content/drive/MyDrive/*/ie643_colab*.zip')
    assert zips, "ie643_colab.zip not found in My Drive: upload it to the My Drive root, wait until the upload has FINISHED, and check this is the same Google account"
    print('using', zips[0], round(os.path.getsize(zips[0]) / 1e6), 'MB')
    !unzip -q "{zips[0]}" -d /content/ie643
%cd /content/ie643
assert os.path.exists('configs/ltx2b.yaml'), "the unzip is incomplete or old: run  !rm -rf /content/ie643  and re-run this cell (NOT anything under /content/drive)"
!echo clips: $(ls data/dataset_got10k/train | wc -l) train, $(ls data/dataset_got10k/val | wc -l) val, $(find data/dataset_got10k -name caption.txt | wc -l) captions


In [ ]:
# Colab already has a matching torch; install the rest (transformers is pinned <5 in requirements.txt).
!grep -v -E '^(torch|gradio)' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

In [ ]:
# Optional: Hugging Face login (faster, more reliable download). The token is typed, never printed or saved to a file.
import getpass
from huggingface_hub import login
tok = getpass.getpass('HF token (blank to skip): ')
if tok: login(tok)
del tok

In [ ]:
# Sanity check of the code on the tiny model (about a minute, no big download).
!python -m pytest tests -q

## Stage 1: does plain 2B hold together with its own prompt? And what does the training-free drag-video start give?
Rows = clip x variant, columns = first / middle / last frame, the target path drawn on each frame.
`base` = plain LTX (no adapter). `ref0.8` / `ref0.65` start the denoising from the drag video (the object's cut-out moved along the
path) noised to sigma 0.8 / 0.65: lower = follows the drag video more closely. `comp2.5` additionally replaces everything outside the object's footprint with the object-free scene: it removes the ghost copy of the original object that `ref0.8` alone leaves at the start.
The adapter is untrained here, so this is the training-free effect alone.

In [ ]:
import os
CFG = '--config configs/ltx2b.yaml'
OUT1 = '/content/drive/MyDrive/ie643_ckpt/stage1_grid'
os.makedirs(OUT1, exist_ok=True)
!python scripts/demo_grid.py {CFG} --num 3 --direction right --variants base,ref0.8,ref0.8+comp2.5,ref0.65+comp2.5 --out {OUT1} 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart"
from IPython.display import Image, display
display(Image(f'{OUT1}/grid.png'))

## Stage 2: train the adapter
Per-clip prompts, drag-video condition, object-only injection and the focus loss are all on (see `configs/ltx2b.yaml`).
Watch `val_obj_gain` (should rise above 0 and keep rising) and `val_bg_gain` (should stay near 0).

In [ ]:
import os
CFG = '--config configs/ltx2b.yaml'
RUN = 'b2_main'          # <- new name for every new experiment
CK = f'/content/drive/MyDrive/ie643_ckpt/{RUN}'
os.makedirs(CK, exist_ok=True)
resume = f'--resume {CK}/last.pt' if os.path.exists(f'{CK}/last.pt') else ''
!python -u scripts/train.py {CFG} --save_dir {CK} {resume} 2>&1 | tee -a {CK}/train_stdout.log

In [ ]:
# Did the adapter learn to use the trajectory?
import json
for l in open(f'{CK}/log.jsonl'):
    r = json.loads(l)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()
           if k in ('epoch', 'val_loss', 'cond_gain', 'val_obj_gain', 'val_bg_gain', 'inject_ratio')})

## Stage 3: the trained adapter, same grid
`adapter` = trained adapter alone, `adapter+mg3` amplifies the trajectory 3x, `ref0.8` adds the drag-video start, `comp2.5` removes the old copy of the object.
Change `--direction` to `left`, `up`, `down` to look at the opposite paths (the instructor saw forward better than backward).

In [ ]:
OUT3 = f'{CK}/grid_trained'
os.makedirs(OUT3, exist_ok=True)
!python scripts/demo_grid.py {CFG} --checkpoint {CK}/best.pt --num 3 --direction right --variants base,adapter,ref0.8,ref0.8+comp2.5 --out {OUT3} 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart"
display(Image(f'{OUT3}/grid.png'))

One example traced from input to result, for the report: input image + object mask + drawn path, the drag video (the condition), plain LTX and the adapter with the SAM 2 mask and tracked path, and the metrics with what each measures. Change `--clip` (a val clip name) and `--direction` to trace other examples.

In [ ]:
!python scripts/trace_example.py {CFG} --checkpoint {CK}/best.pt --direction right --ref_init 0.8 --bg_comp 2.5 --out {CK}/trace 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart|it/s"
display(Image(f'{CK}/trace/trace.png'))

## Stage 4: benchmark (metrics)
Three variants of the adapter against plain LTX on the same images and seeds:
**A** adapter alone, **B** + drag-video start (`--ref_init 0.8`), **C** + scene composite (`--bg_comp 2.5`: no ghost of the original object).
Read `summary.md`: `obj_ate` / `direction_cos` / `progress` = did the object follow the path; `bg_flow` / `bg_drift` = unintended background motion
(lower is better; compare with the baseline row); the per-motion table shows left vs right and up vs down; the last table is the spread over seeds.
Set `FAST = True` in the next cell for a short run; for the report use more images (`NUM_IMAGES = 8`) and `SEEDS = '0,1,2'`.

In [ ]:
FAST = False   # True: 3 images, 1 seed, left/right only (about 15 min for all three variants)
NUM_IMAGES, SEEDS, MOTIONS = (3, '0', 'left,right') if FAST else (4, '0,1', 'left,right,up,down')
BENCH = f'{CK}/bench'
for name, flags in (('A_adapter', ''), ('B_ref0.8', '--ref_init 0.8'), ('C_ref0.8_comp2.5', '--ref_init 0.8 --bg_comp 2.5')):
    !python scripts/benchmark.py {CFG} --checkpoint {CK}/best.pt --num_images {NUM_IMAGES} --seeds {SEEDS} --motions {MOTIONS} {flags} --out {BENCH}/{name} 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart|it/s"
    print('=====', name); print(open(f'{BENCH}/{name}/summary.md').read())

## Stage 5 (optional): clean out-of-domain images
Put clean single-object images (jpg/png) in `My Drive/ood_images`. Optional per image: `name.txt` (your own prompt) and `name_mask.png` (white = object).
Without them the cell below writes a prompt with Qwen2-VL and SAM finds the object from a click at the image centre, so check the masks in the montage.

In [ ]:
OOD = '/content/drive/MyDrive/ood_images'
if os.path.isdir(OOD):
    !python scripts/caption_clips.py --models qwen --write --images_dir {OOD}
    !python scripts/benchmark.py {CFG} --checkpoint {CK}/best.pt --num_images 0 --ood_dir {OOD} --seeds {SEEDS} --motions {MOTIONS} --ref_init 0.8 --bg_comp 2.5 --out {BENCH}/ood 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart|it/s"
    print(open(f'{BENCH}/ood/summary.md').read()); display(Image(f'{BENCH}/ood/montage.png'))
else:
    print('no folder', OOD)

## Stage 6 (optional): the instructor's experiment, focus loss on vs. off
Same data, same settings, same seeds; only `--focus_loss 0` differs (plain mean over all tokens instead of the object/background split).
Question: does the focus loss reduce unintended background movement while the object still follows the path?

In [ ]:
RUN2 = 'b2_nofocus'
CK2 = f'/content/drive/MyDrive/ie643_ckpt/{RUN2}'
os.makedirs(CK2, exist_ok=True)
resume2 = f'--resume {CK2}/last.pt' if os.path.exists(f'{CK2}/last.pt') else ''
!python -u scripts/train.py {CFG} --focus_loss 0 --save_dir {CK2} {resume2} 2>&1 | tee -a {CK2}/train_stdout.log
!python scripts/benchmark.py {CFG} --checkpoint {CK2}/best.pt --num_images {NUM_IMAGES} --seeds {SEEDS} --motions {MOTIONS} --out {CK2}/bench 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart|it/s"

In [ ]:
# Side by side (adapter alone, same images / paths / seeds): object error and background motion reported separately.
import json
def load(path): return json.load(open(path))['summary']['overall']
with_focus, without = load(f'{BENCH}/A_adapter/results.json'), load(f'{CK2}/bench/results.json')
keys = ['ate', 'direction_cos', 'progress', 'obj_ate', 'obj_direction_cos', 'bg_drift', 'bg_flow', 'clip_image_sim']
print(f"{'metric':<20}{'plain LTX':>12}{'focus loss':>12}{'no focus':>12}")
for k in keys:
    print(f"{k:<20}{with_focus['baseline'].get(k, float('nan')):>12.4f}{with_focus['adapter'].get(k, float('nan')):>12.4f}{without['adapter'].get(k, float('nan')):>12.4f}")

## Stage 7 (optional): train on the situation inference uses
With `--ref_init` the model starts from the NOISED DRAG VIDEO, but plain training only shows it noised real video. `--sde_prob 0.5` builds half of the training samples from the noised drag video (target: the velocity towards the real clip), so the adapter learns to repaint pasted objects into real ones. Compare with Stage 4 using the same benchmark flags (`--ref_init 0.8 --bg_comp 2.5`).

In [ ]:
RUN3 = 'b2_sde'
CK3 = f'/content/drive/MyDrive/ie643_ckpt/{RUN3}'
os.makedirs(CK3, exist_ok=True)
resume3 = f'--resume {CK3}/last.pt' if os.path.exists(f'{CK3}/last.pt') else ''
!python -u scripts/train.py {CFG} --sde_prob 0.5 --save_dir {CK3} {resume3} 2>&1 | tee -a {CK3}/train_stdout.log
!python scripts/benchmark.py {CFG} --checkpoint {CK3}/best.pt --num_images {NUM_IMAGES} --seeds {SEEDS} --motions {MOTIONS} --ref_init 0.8 --bg_comp 2.5 --out {CK3}/bench 2>&1 | grep --line-buffered -v -i -E "warning|onednn|absl|cudart|it/s"
print(open(f'{CK3}/bench/summary.md').read())

## Stage 8: check a checkpoint's adapter strength (no retraining)
`as0` = the adapter switched off (training-free path, same checkpoint loaded); `as0.3` / `as0.6` = fractions of the trained strength; `until0.4` = adapter only while sigma >= 0.4. If `as0.3` looks clean and follows the path better than `as0`, the adapter helps but was too strong.

In [ ]:
CKP = '/content/drive/MyDrive/ie643_ckpt/b2_sde2/last.pt'   # any checkpoint
!python scripts/demo_grid.py {CFG} --num 3 --checkpoint {CKP} --variants ref0.8+comp2.5+as0,ref0.8+comp2.5+as0.3,ref0.8+comp2.5+as0.6,ref0.8+comp2.5+until0.4 --out /content/grid_strength
